In [1]:
import duckdb
import pandas as pd
# Charger l'extension SQL magique
%load_ext sql

# Établir la connexion avec DuckDB
# Connexion en mémoire (temporaire)
%sql duckdb:///

# Ou connexion persistante (optionnel)
# %sql duckdb:///ma_base.duckdb
# Configuration complète de JupySQL
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = 1  # Affiche le nombre de lignes et le temps
%config SqlMagic.displaycon = False

Connecting to 'duckdb:///'

In [3]:
%%sql
-- Créer des tables à partir des fichiers CSV
CREATE OR REPLACE TABLE data AS 
SELECT * FROM read_csv_auto('retail_sales_data.csv');

-- Vérifier que les tables sont créées
SHOW TABLES;

,Success


In [4]:
%%sql
SELECT * 
FROM data

,Order_ID,Order_Date,Product,Category,Quantity,Price,Revenue,Profit,Region
0,1,2019-03-13,Apple Watch,Accessories,4,1360,5440,1926.53,North
1,2,2020-04-03,Sony Headphones,Accessories,2,82886,165772,62165.54,West
2,3,2020-10-23,Sony Headphones,Accessories,4,60763,243052,54196.19,North
3,4,2019-07-11,Samsung Galaxy S23,Electronics,4,65425,261700,90787.71,West
4,5,2019-09-10,Sony Headphones,Accessories,1,54207,54207,13393.83,West
...,...,...,...,...,...,...,...,...,...
4995,4996,2020-09-28,iPhone 14,Electronics,5,38307,191535,73202.34,South
4996,4997,2020-06-07,Dell Laptop,Electronics,2,2792,5584,1217.83,South
4997,4998,2019-12-27,iPad,Electronics,4,1780,7120,2712.66,East
4998,4999,2020-04-18,Puma T-shirt,Fashion,3,69050,207150,52515.78,North


In [ ]:
%%sql
-- Afficher les années uniques présentes dans les données
SELECT YEAR(Order_Date) AS Annee
FROM data 
GROUP BY Annee


,Annee
0,2019
1,2020


In [ ]:
%%sql
-- Calculer le revenu total par année
SELECT YEAR(Order_Date) AS Annee,SUM(Revenue) AS Revenue_Total
FROM data 
GROUP BY Annee

,Annee,Revenue_Total
0,2019,327593029.0
1,2020,331998966.0


In [ ]:
%%sql
-- Calculer le revenu total par année et par produit
SELECT Product,YEAR(Order_Date) AS Annee,SUM(Revenue) AS Revenue_Total
FROM data 
GROUP BY Annee,Product
LIMIT 10

,Product,Annee,Revenue_Total
0,Sony Headphones,2020,31137271.0
1,Dell Laptop,2020,25166605.0
2,HP Laptop,2019,26702064.0
3,Boat Earbuds,2019,30464367.0
4,Sony Headphones,2019,25705355.0
5,Dell Laptop,2019,30104161.0
6,Boat Earbuds,2020,24592472.0
7,HP Laptop,2020,29482849.0
8,Canon Camera,2020,28925425.0
9,Samsung Galaxy S23,2020,26978609.0


In [ ]:
%%sql
-- Calculer le revenu total par année et par produit, et classer les produits par revenu pour chaque année
SELECT YEAR(Order_Date) AS Annee,Product,SUM(Revenue) AS Revenue_Total,
                DENSE_RANK()
                OVER(PARTITION BY YEAR(Order_Date) ORDER BY SUM(Revenue)DESC ) AS Rang

FROM data 
GROUP BY Annee,Product
LIMIT 10

,Annee,Product,Revenue_Total,Rang
0,2019,Adidas Shoes,31175329.0,1
1,2019,Boat Earbuds,30464367.0,2
2,2019,Dell Laptop,30104161.0,3
3,2019,Canon Camera,29221780.0,4
4,2019,Nike Shoes,28610967.0,5
5,2019,Puma T-shirt,27134967.0,6
6,2019,HP Laptop,26702064.0,7
7,2019,Apple Watch,26443544.0,8
8,2019,Sony Headphones,25705355.0,9
9,2019,iPhone 14,25041652.0,10


In [ ]:
%%sql
--  Les 3 meuleurs vente par Année
WITH CTE AS (
SELECT YEAR(Order_Date) AS Annee,Product,SUM(Revenue) AS Revenue_Total,
                DENSE_RANK()
                OVER(PARTITION BY YEAR(Order_Date) ORDER BY SUM(Revenue)DESC ) AS Rang

FROM data 
GROUP BY Annee,Product
)
SELECT *
FROM CTE
WHERE Rang BETWEEN 1 AND 3


,Annee,Product,Revenue_Total,Rang
0,2019,Adidas Shoes,31175329.0,1
1,2019,Boat Earbuds,30464367.0,2
2,2019,Dell Laptop,30104161.0,3
3,2020,iPhone 14,31544872.0,1
4,2020,Sony Headphones,31137271.0,2
5,2020,iPad,29919020.0,3


In [ ]:
%%sql
-- Calculer le revenu total par catégorie et par produit, et classer les produits par revenu pour chaque catégorie
SELECT Category,Product,SUM(Revenue) AS Revenue_Total,
                DENSE_RANK()
                OVER(PARTITION BY Category ORDER BY SUM(Revenue)DESC ) AS Classement

FROM data 
GROUP BY Category,Product
LIMIT 10

,Category,Product,Revenue_Total,Classement
0,Electronics,Canon Camera,58147205.0,1
1,Electronics,iPhone 14,56586524.0,2
2,Electronics,HP Laptop,56184913.0,3
3,Electronics,Dell Laptop,55270766.0,4
4,Electronics,iPad,53452232.0,5
5,Electronics,Samsung Galaxy S23,50434240.0,6
6,Fashion,Adidas Shoes,56611473.0,1
7,Fashion,Nike Shoes,54891577.0,2
8,Fashion,Puma T-shirt,54694636.0,3
9,Accessories,Sony Headphones,56842626.0,1


In [ ]:
%%sql
--  Les 3 meuleurs vente par categorie
WITH CTE AS (
SELECT Category,Product,SUM(Revenue) AS Revenue_Total,
                DENSE_RANK()
                OVER(PARTITION BY Category ORDER BY SUM(Revenue)DESC ) AS Classement

FROM data 
GROUP BY Category,Product
)
SELECT *
FROM CTE
WHERE Classement BETWEEN 1 AND 3

,Category,Product,Revenue_Total,Classement
0,Accessories,Sony Headphones,56842626.0,1
1,Accessories,Boat Earbuds,55056839.0,2
2,Accessories,Apple Watch,51418964.0,3
3,Electronics,Canon Camera,58147205.0,1
4,Electronics,iPhone 14,56586524.0,2
5,Electronics,HP Laptop,56184913.0,3
6,Fashion,Adidas Shoes,56611473.0,1
7,Fashion,Nike Shoes,54891577.0,2
8,Fashion,Puma T-shirt,54694636.0,3


In [ ]:
%%sql
-- Calculer le revenu total par région et par produit, et classer les produits par revenu pour chaque région
SELECT Region,Product,SUM(Revenue) AS Revenue_Total,
                DENSE_RANK()
                OVER(PARTITION BY Region ORDER BY SUM(Revenue)DESC ) AS Classement

FROM data 
GROUP BY Region,Product
LIMIT 10

,Region,Product,Revenue_Total,Classement
0,South,Sony Headphones,16437178.0,1
1,South,Canon Camera,15594728.0,2
2,South,iPhone 14,15224304.0,3
3,South,Nike Shoes,14772852.0,4
4,South,Puma T-shirt,14672622.0,5
5,South,Boat Earbuds,14117371.0,6
6,South,Adidas Shoes,14057671.0,7
7,South,HP Laptop,13967836.0,8
8,South,Samsung Galaxy S23,13531189.0,9
9,South,Apple Watch,13403095.0,10


In [29]:
%%sql
--  Les 3 meuleurs vente par categorie
WITH CTE AS (
SELECT Region,Product,SUM(Revenue) AS Revenue_Total,
                DENSE_RANK()
                OVER(PARTITION BY Region ORDER BY SUM(Revenue)DESC ) AS Classement

FROM data 
GROUP BY Region,Product
)
SELECT *
FROM CTE
WHERE Classement BETWEEN 1 AND 5

,Region,Product,Revenue_Total,Classement
0,West,Dell Laptop,16084206.0,1
1,West,Puma T-shirt,15758902.0,2
2,West,Canon Camera,14715107.0,3
3,West,Adidas Shoes,14107605.0,4
4,West,iPhone 14,14103588.0,5
5,South,Sony Headphones,16437178.0,1
6,South,Canon Camera,15594728.0,2
7,South,iPhone 14,15224304.0,3
8,South,Nike Shoes,14772852.0,4
9,South,Puma T-shirt,14672622.0,5


In [ ]:
%%sql
-- Calculer le nombre total de commandes par région et par produit, et classer les produits par nombre de commandes pour chaque région
SELECT Region,Product,COUNT(DISTINCT Order_ID) AS Revenue_Total,
                DENSE_RANK()
                OVER(PARTITION BY Region ORDER BY COUNT(DISTINCT Order_ID)DESC ) AS Classement

FROM data 
GROUP BY Region,Product
LIMIT 10

,Region,Product,Revenue_Total,Classement
0,South,HP Laptop,114,1
1,South,Sony Headphones,114,1
2,South,Puma T-shirt,113,2
3,South,Samsung Galaxy S23,111,3
4,South,Nike Shoes,108,4
5,South,Apple Watch,107,5
6,South,iPhone 14,105,6
7,South,Canon Camera,104,7
8,South,Dell Laptop,103,8
9,South,Boat Earbuds,102,9


In [33]:
%%sql
--  le  top 5 vente par  region
WITH CTE AS (
SELECT Region,Product,COUNT(DISTINCT Order_ID) AS Nombre_vente,
                DENSE_RANK()
                OVER(PARTITION BY Region ORDER BY COUNT(DISTINCT Order_ID)DESC ) AS Classement

FROM data 
GROUP BY Region,Product
)
SELECT *
FROM CTE
WHERE Classement BETWEEN 1 AND 5

,Region,Product,Nombre_vente,Classement
0,West,Puma T-shirt,121,1
1,West,Dell Laptop,113,2
2,West,Sony Headphones,108,3
3,West,Canon Camera,104,4
4,West,iPhone 14,104,4
5,West,Samsung Galaxy S23,103,5
6,West,Adidas Shoes,103,5
7,South,Sony Headphones,114,1
8,South,HP Laptop,114,1
9,South,Puma T-shirt,113,2


In [ ]:
%%sql
-- Calculer le nombre total de commandes par catégorie et par produit, et classer les produits par nombre de commandes pour chaque catégorie
SELECT Category,Product,COUNT(DISTINCT Order_ID) AS Nombre_vente,
                DENSE_RANK()
                OVER(PARTITION BY Category ORDER BY COUNT(DISTINCT Order_ID)DESC ) AS Classement

FROM data 
GROUP BY Category,Product
LIMIT 10

,Category,Product,Nombre_vente,Classement
0,Electronics,iPhone 14,426,1
1,Electronics,iPad,421,2
2,Electronics,HP Laptop,417,3
3,Electronics,Canon Camera,416,4
4,Electronics,Dell Laptop,413,5
5,Electronics,Samsung Galaxy S23,394,6
6,Fashion,Puma T-shirt,441,1
7,Fashion,Nike Shoes,428,2
8,Fashion,Adidas Shoes,394,3
9,Accessories,Sony Headphones,433,1


In [ ]:
%%sql
-- le  top 5 vente par  categorie
WITH CTE AS (
SELECT Category,Product,COUNT(DISTINCT Order_ID) AS Nombre_vente,
                DENSE_RANK()
                OVER(PARTITION BY Category ORDER BY COUNT(DISTINCT Order_ID)DESC ) AS Classement

FROM data 
GROUP BY Category,Product
)
SELECT*
FROM CTE
WHERE Classement BETWEEN 1 AND 5

,Category,Product,Nombre_vente,Classement
0,Electronics,iPhone 14,426,1
1,Electronics,iPad,421,2
2,Electronics,HP Laptop,417,3
3,Electronics,Canon Camera,416,4
4,Electronics,Dell Laptop,413,5
5,Fashion,Puma T-shirt,441,1
6,Fashion,Nike Shoes,428,2
7,Fashion,Adidas Shoes,394,3
8,Accessories,Sony Headphones,433,1
9,Accessories,Boat Earbuds,415,2
